# 🥉 Camada Bronze — Ingestão de Dados Brutos

## O que é a camada Bronze?
A Bronze é a **primeira camada** da arquitetura Medallion. Seu único objetivo é **preservar o dado exatamente como ele chegou** da fonte — sem qualquer transformação de conteúdo.

## Por que não tratar os dados direto aqui?
Imagine que você aplique uma regra de limpeza errada e descarte dados importantes. Se você não guardou o raw original, **perdeu os dados para sempre**. A Bronze é o seu *ponto de restauração*. Qualquer reprocessamento começa por ela.

### O que fazemos nesta camada:
| Ação | Motivo |
|---|---|
| Ler CSV como `dtype=str` | Não inferir tipos — preservar o dado original |
| Adicionar `_source_file` | Rastreabilidade: saber de onde veio cada linha |
| Adicionar `_ingested_at` | Auditoria: saber quando foi ingerido |
| Adicionar `_row_hash` | Detectar mudanças em reprocessamentos futuros |
| Salvar em **Parquet** | Compressão eficiente, mantém schema, mais rápido que CSV |

In [ ]:
# ── Configuração para Google Colab ────────────────────────────────────────────
# Se estiver no Colab, monte o Drive e clone/suba seu projeto.
# Descomente as linhas abaixo:

# from google.colab import drive
# drive.mount('/content/drive')
# import os
# os.chdir('/content/drive/MyDrive/meu-projeto-medalhao')

# ── Instalação de dependências (apenas Colab) ─────────────────────────────────
# !pip install -q pandas pyarrow

import sys
sys.path.insert(0, '..')  # permite importar src/utils.py

import pandas as pd
from pathlib import Path
from datetime import datetime
import hashlib

print('Dependências carregadas com sucesso.')

## 1. Verificar arquivos na Landing Zone
A **Landing Zone** é onde os arquivos chegam — pode ser um S3, SFTP, ou pasta local.
Aqui usamos a pasta `landing/` com arquivos CSV simulando dados de um ERP (SAP-like).

In [ ]:
landing_path = Path('../landing')

arquivos = list(landing_path.glob('*.csv'))
print(f'Arquivos encontrados na Landing Zone: {len(arquivos)}')
for f in arquivos:
    print(f'  → {f.name}  ({f.stat().st_size} bytes)')

## 2. Inspecionar o dado bruto
Antes de ingerir, **sempre inspecionamos** o arquivo para entender sua estrutura.
Isso é o equivalente a abrir o arquivo no Excel antes de processar — mas fazemos via código.

In [ ]:
# Leitura exploratória — ainda NÃO é a ingestão oficial
df_raw_1 = pd.read_csv('../landing/z0019_1.csv', sep=';', dtype=str)
df_raw_2 = pd.read_csv('../landing/z0019_2.csv', sep=';', dtype=str)

print('=== Arquivo 1 ===')
print(f'Shape: {df_raw_1.shape}')
display(df_raw_1.head())

print('\n=== Arquivo 2 ===')
print(f'Shape: {df_raw_2.shape}')
display(df_raw_2.head())

### Legenda das colunas (domínio de negócio)
Estes dados simulam um extrato de estoque de um ERP:

| Coluna | Descrição |
|---|---|
| `NATB` | Número do material (chave primária) |
| `MAKTX` | Descrição do material |
| `WERKS` | Centro (planta/armazém) |
| `MAINS` | Tipo de estoque |
| `LABST` | Estoque disponível (unrestricted) |

In [ ]:
# Verificar tipos inferidos (todos devem ser 'object' pois lemos como string)
print('Tipos do arquivo 1:')
print(df_raw_1.dtypes)

print('\nVerificar valores únicos por coluna:')
for col in df_raw_1.columns:
    print(f'  {col}: {df_raw_1[col].nunique()} valores únicos → {df_raw_1[col].unique().tolist()}')

## 3. Função de Ingestão Bronze
Agora definimos a função que realiza a **ingestão oficial** — adiciona metadados de auditoria
e persiste em Parquet. Note que **nenhum dado é alterado**.

In [ ]:
def ingest_to_bronze(filepath: str, sep: str = ';') -> pd.DataFrame:
    """
    Ingere um CSV para a camada Bronze sem transformar o conteúdo.
    Adiciona apenas colunas de auditoria (_source_file, _ingested_at, _row_hash).
    """
    src = Path(filepath)
    
    # dtype=str é INTENCIONAL: não queremos que pandas interprete '0019' como 19
    df = pd.read_csv(src, sep=sep, dtype=str)
    
    # Metadados de auditoria — não alteram o dado de negócio
    df['_source_file'] = src.name
    df['_ingested_at'] = datetime.utcnow().isoformat()
    df['_row_hash'] = df.apply(
        lambda row: hashlib.md5(str(row.values).encode('utf-8')).hexdigest(),
        axis=1
    )
    
    # Persistência em Parquet
    bronze_dir = Path('../data/bronze')
    bronze_dir.mkdir(parents=True, exist_ok=True)
    
    dest = bronze_dir / src.name.replace('.csv', '.parquet')
    df.to_parquet(dest, index=False)
    
    print(f'[Bronze] ✓ {src.name} → {dest.name} | {len(df)} linhas | {dest.stat().st_size} bytes')
    return df

In [ ]:
# Ingestão dos dois arquivos
df_bronze_1 = ingest_to_bronze('../landing/z0019_1.csv')
df_bronze_2 = ingest_to_bronze('../landing/z0019_2.csv')

print('\n--- Estado dos dados na camada Bronze ---')
print('Todos os valores são strings. Metadados de auditoria adicionados.')
print(f'Schema: {df_bronze_1.dtypes.to_dict()}')

## 4. Head() — Visualização do Estado Bronze
> **O que mudou?** Os dados são **idênticos** ao CSV original, mas agora estão em Parquet
> com três colunas extras de auditoria. Nenhum valor de negócio foi alterado.

In [ ]:
print('=== HEAD — Bronze (arquivo 1) ===')
display(df_bronze_1.head())

print('\n=== HEAD — Bronze (arquivo 2) ===')
display(df_bronze_2.head())

print('\n=== Verificação: nenhum nulo foi introduzido ===')
print(df_bronze_1.isnull().sum())

## Resumo da Camada Bronze

| Métrica | Valor |
|---|---|
| Arquivos ingeridos | 2 |
| Linhas totais | 6 |
| Transformações no conteúdo | **Nenhuma** |
| Colunas de auditoria adicionadas | 3 (`_source_file`, `_ingested_at`, `_row_hash`) |
| Formato de saída | Parquet |

**Próximo passo:** Notebook `02_processamento_silver.ipynb` — limpeza, tipagem e deduplicação.